# Membrane Protein RBFE with packmol-memgen and OpenFE

This tutorial walks through setting up a Relative Binding Free Energy (RBFE) simulation for a membrane-embedded protein using:
- `packmol-memgen`: to build the lipid bilayer system
- `OpenMM`: to check if we can run a few steps of MD with the system
- `openfe`: to define and run the RBFE network

**System**: A2A adenosine receptor (GPCR) in a POPC membrane

**Ligands**: Congeneric small molecules whose relative binding affinities we wish to predict

## Imports

In [1]:
import pathlib
import re
import subprocess
import xml.etree.ElementTree as ET
from collections import defaultdict

import openmmforcefields
from openmm import (
    LangevinMiddleIntegrator,
    MonteCarloMembraneBarostat,
    unit,
)
from openmm.app import (
    ForceField, HBonds, PDBFile, PME, Simulation, Topology,
)

## Contents

1. [System preparation with packmol-memgen](#1-system-preparation-with-packmol-memgen)
2. [Fix common PDB issues and prepare the topology](#2-fix-common-pdb-issues-and-prepare-the-topology)
3. [Running a short MD smoke test](#3-running-a-short-md-smoke-test)
4. [Set up and run the RBFE campaign](#4-set-up-and-run-the-rbfe-campaign)
5. [Going further: the Python API](#5-going-further-the-python-api)

## 1. System preparation with packmol-memgen

We use `packmol-memgen` to embed the A2A receptor into a POPC bilayer.  
The receptor PDB (`a2a/a2a.pdb`) should already be:
- Oriented along the membrane normal (PPM server or OPM database)
- Protonation states assigned

> **Note:** The two flags below are easy to get wrong and will silently
> produce an incorrect system if misused:
>
> - `--preoriented`: only pass this if your PDB was downloaded from the
>   OPM database or has already been oriented by the PPM server. If your
>   structure has not been pre-oriented, omit this flag and packmol-memgen
>   will orient it automatically.
> - `--notprotonate`: only pass this if you have already assigned
>   protonation states yourself. If
>   omitted, packmol-memgen will add hydrogens, which may not match your
>   intended protonation state.

In [2]:
# !packmol-memgen \
#     --pdb a2a/a2a.pdb \
#     --lipids POPC \
#     --preoriented \
#     --dist 17 \
#     --dist_wat 15 \
#     --salt \
#     --salt_c K+ \
#     --saltcon 0.15 \
#     --nottrim \
#     --overwrite \
#     --notprotonate \
#     --charmm

PMEMD not found. Setting path to SANDER


 _____           _                    _
|  __ \         | |                  | |
| |__) |_ _  ___| | ___ __ ___   ___ | |
|  ___/ _` |/ __| |/ / '_ ` _ \ / _ \| |
| |  | (_| | (__|   <| | | | | | (_) | |
|_|   \__,_|\___|_|\_\_| |_| |_|\___/|_|
                                         ___
                  /\/\   ___ _ __ ___   / _ \___ _ __
                 /    \ / _ \ '_ ` _ \ / /_\/ _ \ '_ \ 
                / /\/\ \  __/ | | | | / /_ \  __/ | | |
                \/    \/\___|_| |_| |_\____/\___|_| |_| 


###############################################################
Stephan Schott-Verdugo 2016-11-07        VERSION: 2025.1.29
Generated at CPCLab at Heinrich Heine University Duesseldorf
      &      CBCLab at Forschungszentrum Juelich
###############################################################

--verbose for details of the packing process
No ratio provided (--ratio).
Preprocessing a2a/a2a.pdb. This might take a minute.

     Information

### Key flags explained

| Flag | Purpose |
|---|---|
| `--preoriented` | Skip PPM alignment; the PDB is already membrane-oriented |
| `--dist 17` | ~17 Å padding between protein and periodic box edge in the xy-plane |
| `--dist_wat 15` | Water layer thickness above/below the bilayer |
| `--nottrim` | Prevents clipping of lipid tails that partially overlap the protein |
| `--charmm` | Write one residue per lipid molecule (CHARMM convention), required for compatibility with Lipid17 force field templates |
| `--notprotonate` | Skip internal protonation (protonation states have already been assigned) |
| `--salt` | Add counterions to neutralise the system |
| `--salt_c K+` | Use K⁺ as cation |
| `--saltcon 0.15` | Salt concentration of 150 mM, matching physiological ionic strength |

The main output we need is **`bilayer_a2a.pdb`**.  
`packmol-memgen` also writes a log file containing the box vectors, which we will need in the next step.

## 2. Fix common PDB issues and prepare the topology

### 2a. Parse box dimensions from the log file and patch the CRYST1 record

`packmol-memgen` writes box dimensions to the log file but not to the 
[`CRYST1`](https://www.wwpdb.org/documentation/file-format-content/format33/sect8.html#CRYST1) 
record. We extract the dimensions from the log and write them into the PDB file.

In [3]:
def parse_box_from_log(log_path: str) -> tuple[float, float, float]:
    """
    Extract box dimensions (Å) from a packmol-memgen log file.
    
    Looks for lines of the form:
        x_len = 88.52550719674292
        y_len = 88.52550719674292
        z_len = 108.602
    """
    dims = {}
    pattern = re.compile(r"(x_len|y_len|z_len)\s*=\s*(\d+\.\d+)")
    with open(log_path) as fh:
        for line in fh:
            m = pattern.search(line)
            if m:
                dims[m.group(1)] = float(m.group(2))
            if len(dims) == 3:
                break
    if len(dims) < 3:
        raise ValueError(
            f"Could not find x_len/y_len/z_len in {log_path}. "
            f"Found: {dims}"
        )
    return dims["x_len"], dims["y_len"], dims["z_len"]

def patch_cryst1(pdb_in: str, pdb_out: str, a: float, b: float, c: float) -> None:
    """Write / overwrite the CRYST1 record with correct box vectors."""
    cryst1 = f"CRYST1{a:9.3f}{b:9.3f}{c:9.3f}  90.00  90.00  90.00 P 1           1\n"
    lines = open(pdb_in).readlines()
    lines = [l for l in lines if not l.startswith("CRYST1")]
    lines.insert(0, cryst1)
    with open(pdb_out, "w") as fh:
        fh.writelines(lines)
    print(f"Written {pdb_out} with CRYST1: a={a}, b={b}, c={c}")

In [4]:
a, b, c = parse_box_from_log("a2a/packmol-memgen.log")
patch_cryst1("a2a/bilayer_a2a.pdb", "a2a/bilayer_a2a_fixed.pdb", a, b, c)

Written a2a/bilayer_a2a_fixed.pdb with CRYST1: a=88.52550719674292, b=88.52550719674292, c=108.602


### 2b. Fix the element column for ions

The last column of the PDB ATOM/HETATM record may contain a single-character element symbol instead of the correct two-character one. For example, the sodium ion line looks like:

`ATOM   4654  Na  NA  A 298  -2.406  -0.748  -2.531  1.00  0.00           N`

The element column reads N instead of NA, causing OpenMM to misidentify the ion as nitrogen. We will fix this here:

In [5]:
def fix_ion_elements(pdb_in: str, pdb_out: str) -> None:
    """Correct the element column (cols 77-78) for ions that packmol-memgen
    writes with a truncated element symbol.

    Args:
        pdb_in:  Path to the input PDB file.
        pdb_out: Path to the output PDB file.
    """
    # (residue_name, atom_name) → correct two-character element string
    fixes = {
        ("NA", "Na"): "NA",
        ("CL", "Cl"): "CL",
        ("K+", "K" ): " K",
        ("MG", "Mg"): "MG",
    }
    corrected = 0
    out_lines = []
    with open(pdb_in) as fh:
        for line in fh:
            if line.startswith(("ATOM", "HETATM")) and len(line) >= 78:
                res_name  = line[17:20].strip()
                atom_name = line[12:16].strip()
                key = (res_name, atom_name)
                if key in fixes and line[76:78].strip() != fixes[key].strip():
                    line = line[:76] + fixes[key] + line[78:]
                    corrected += 1
            out_lines.append(line)
    with open(pdb_out, "w") as fh:
        fh.writelines(out_lines)
    print(f"Fixed element column for {corrected} ion records → {pdb_out}")

In [6]:
fix_ion_elements("a2a/bilayer_a2a_fixed.pdb", "a2a/bilayer_a2a_fixed.pdb")

Fixed element column for 1 ion records → a2a/bilayer_a2a_fixed.pdb


### 2c. Add lipid bond information and prepare the topology

`packmol-memgen` does not write [`CONECT`](https://www.wwpdb.org/documentation/file-format-content/format33/sect10.html#CONECT) 
records for lipid residues, which is a [known issue](https://github.com/openmm/openmm/issues/4997). 
Lipids are heterogens and are required by the PDB spec to use `CONECT` records, files that omit them are nonstandard.

The fix is to load the bond definitions from the Amber Lipid17 force field XML before reading the PDB, so that 
[`Topology.createStandardBonds()`](https://docs.openmm.org/latest/api-python/generated/openmm.app.topology.Topology.html#openmm.app.topology.Topology.createStandardBonds) 
can apply them. This method inspects each residue in the `openmm.app.Topology` against its registered 
bond templates and adds the appropriate bonds in-place. But it can only do so if the templates have 
been registered first via 
[`Topology.loadBondDefinitions()`](https://docs.openmm.org/latest/api-python/generated/openmm.app.topology.Topology.html#openmm.app.topology.Topology.loadBondDefinitions).

There is one further complication: `lipid17_merged.xml` stores bonds with `atomName1`/`atomName2` 
attributes (the Amber FF format), but `loadBondDefinitions()` expects `from`/`to` attributes 
(the residue template format). The `extract_bond_info()` function below handles this conversion.

In [7]:
_RESNAME_FIXES: dict[str, str] = {
    "POP": "POPC",
}

def extract_bond_info(xml_path: str, output_path: str) -> None:
    """Convert bond definitions from Amber FF XML format to the
    from/to format expected by Topology.loadBondDefinitions().

    Amber FF XML uses:   <Bond atomName1="C1" atomName2="C2"/>
    loadBondDefinitions expects: <Bond from="C1" to="C2"/>

    Args:
        xml_path:    Path to the Amber FF XML (e.g. lipid17_merged.xml).
        output_path: Path to write the converted XML.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    residues = root.find("Residues")
    with open(output_path, "w") as f:
        f.write("<Residues>\n")
        for residue in residues.findall("Residue"):
            res_name = residue.get("name")
            f.write(f'  <Residue name="{res_name}">\n')
            for bond in residue.findall("Bond"):
                atom1 = bond.get("atomName1")
                atom2 = bond.get("atomName2")
                f.write(f'    <Bond from="{atom1}" to="{atom2}"/>\n')
            f.write('  </Residue>\n')
        f.write("</Residues>\n")
    print(f"Bond definitions saved to: {output_path}")


def prepare_membrane_topology(
    input_pdb: str,
    output_pdb: str,
    verbose: bool = True,
) -> None:
    """Prepare a packmol-memgen --charmm PDB for use with OpenMM.

    Topology.loadBondDefinitions() must be called with the converted lipid
    bond XML before calling this function, so that createStandardBonds()
    can find and apply the lipid bond templates.

    This function also renames POP → POPC to match the force field template
    names, since packmol-memgen truncates POPC to POP in CHARMM output.

    Args:
        input_pdb:  Path to the input PDB (with CRYST1 already patched).
        output_pdb: Path to write the corrected PDB.
        verbose:    If True, print a per-residue-type bond count summary.
    """
    pdb = PDBFile(input_pdb)

    if pdb.topology.getPeriodicBoxVectors() is None:
        raise ValueError(
            "No periodic box vectors found. "
            "Run patch_cryst1() before calling prepare_membrane_topology()."
        )

    for res in pdb.topology.residues():
        if res.name in _RESNAME_FIXES:
            res.name = _RESNAME_FIXES[res.name]

    pdb.topology.createStandardBonds()

    with open(output_pdb, "w") as fh:
        PDBFile.writeFile(pdb.topology, pdb.positions, fh, keepIds=True)

    if verbose:
        counts: dict[str, int] = defaultdict(int)
        for bond in pdb.topology.bonds():
            counts[bond[0].residue.name] += 1
        print(f"{'Residue':<8}  {'Total bonds':>12}")
        print("-" * 22)
        for name, count in sorted(counts.items()):
            print(f"{name:<8}  {count:>12}")
        print(f"\nSaved : {output_pdb}")

Now run the full pipeline. The order matters,
[`Topology.loadBondDefinitions()`](https://docs.openmm.org/latest/api-python/generated/openmm.app.topology.Topology.html#openmm.app.topology.Topology.loadBondDefinitions) 
must be called before 
[`PDBFile`](https://docs.openmm.org/latest/api-python/generated/openmm.app.pdbfile.PDBFile.html):

In [8]:
# Find the lipid17 XML bundled with openmmforcefields
ff_xml = str(
    pathlib.Path(openmmforcefields.__file__).parent
    / "ffxml" / "amber" / "lipid17_merged.xml"
)

# Step 1: convert atomName1/atomName2 → from/to format
extract_bond_info(ff_xml, "a2a/lipid17_bonds.xml")

# Step 2: register with OpenMM — must happen before PDBFile() is called
Topology.loadBondDefinitions("a2a/lipid17_bonds.xml")

# Step 3: rename residues, build bonds, write corrected PDB
prepare_membrane_topology(
    input_pdb="a2a/bilayer_a2a_fixed.pdb",
    output_pdb="a2a/bilayer_a2a_prepped.pdb",
)

Bond definitions saved to: a2a/lipid17_bonds.xml


/Users/hannahbaumann/.local/share/mamba/envs/openfe_env/lib/python3.12/site-packages/openmm/app/internal/pdbstructure.py:441: UserWarning: WARNING: two consecutive residues with same number (ATOM   3199  N   NME A 210      17.633   8.568 -23.707  1.00  0.00           N  , ATOM   3198 HD23 LEU A 210      13.213   5.988 -24.001  1.00  0.00           H  )
  warnings.warn("WARNING: two consecutive residues with same number (%s, %s)" % (atom, self._current_residue.atoms[-1]))
/Users/hannahbaumann/.local/share/mamba/envs/openfe_env/lib/python3.12/site-packages/openmm/app/internal/pdbstructure.py:441: UserWarning: WARNING: two consecutive residues with same number (ATOM   3211  N   GLU A 211       9.982   5.977 -38.597  1.00  0.00           N  , ATOM   3210 HH33 ACE A 211      11.950   4.338 -37.817  1.00  0.00           H  )
  warnings.warn("WARNING: two consecutive residues with same number (%s, %s)" % (atom, self._current_residue.atoms[-1]))
/Users/hannahbaumann/.local/share/mamba/envs/ope

Residue    Total bonds
----------------------
ACE                  6
ALA                390
ARG                312
ASN                182
ASP                 48
CYS                150
GLN                136
GLU                105
GLY                128
HIS                110
HOH              32374
ILE                551
LEU                684
LYS                110
MET                102
NME                 10
PHE                399
POPC             28462
PRO                180
SER                165
THR                154
TRP                156
TYR                220
VAL                416

Saved : a2a/bilayer_a2a_prepped.pdb


## 3. Running a short MD smoke test

<div class="alert alert-block alert-warning">
    <b>Warning:</b> The minimisation and short NPT run below only confirm the 
    system is parameterisable and runs without exploding. This is not a 
    production equilibration protocol. A real equilibration should include an 
    NVT warm-up and staged NPT runs with the membrane barostat — the exact 
    protocol will depend on your system, and the steps shown here are just one 
    example approach. Do not use <code>complex_equil.pdb</code> directly as 
    input to an RBFE campaign without proper equilibration.
</div>

In [9]:
pdb = PDBFile('a2a/bilayer_a2a_prepped.pdb')

forcefield = ForceField(
    'amber/protein.ff14SB.xml',
    'amber/lipid17_merged.xml',
    'amber/tip3p_standard.xml',
)

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0 * unit.nanometer,
    constraints=HBonds,
    rigidWater=True,
)

# MonteCarloMembraneBarostat is required for lipid bilayers —
# it allows independent xy and z fluctuations.
# Surface tension of 0 bar·nm = tensionless ensemble.
system.addForce(MonteCarloMembraneBarostat(
    1.0 * unit.bar,
    0.0 * unit.bar * unit.nanometer,
    300 * unit.kelvin,
    MonteCarloMembraneBarostat.XYIsotropic,
    MonteCarloMembraneBarostat.ZFree,
    50,
))

integrator = LangevinMiddleIntegrator(
    300 * unit.kelvin,
    1.0 / unit.picosecond,
    2.0 * unit.femtoseconds,
)

simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.context.setPeriodicBoxVectors(*pdb.topology.getPeriodicBoxVectors())

# Energy minimisation — membrane systems often have clashes from packing
print("Minimizing energy...")
simulation.minimizeEnergy(maxIterations=1000)

# Check the energy is sensible after minimisation — NaN or very large
# values (> 1e7 kJ/mol) indicate something is wrong
state = simulation.context.getState(getEnergy=True)
e = state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
print(f"Potential energy after minimisation: {e:.1f} kJ/mol")
if e > 1e7 or e != e:   # NaN check: NaN != NaN is True
    raise RuntimeError(f"Energy looks wrong after minimisation: {e} kJ/mol")

# Short NPT run  
print("Running short NPT (500 steps)...")
simulation.step(500)

# Save final state
state = simulation.context.getState(getPositions=True, enforcePeriodicBox=True)
positions = state.getPositions()
box_vectors = state.getPeriodicBoxVectors()
pdb.topology.setPeriodicBoxVectors(box_vectors)

with open('a2a/complex_equil.pdb', 'w') as f:
    PDBFile.writeFile(pdb.topology, positions, f)

/Users/hannahbaumann/.local/share/mamba/envs/openfe_env/lib/python3.12/site-packages/openmm/app/internal/pdbstructure.py:441: UserWarning: WARNING: two consecutive residues with same number (HETATM 3199  N   NME A 210      17.633   8.568 -23.707  1.00  0.00           N  , ATOM   3198 HD23 LEU A 210      13.213   5.988 -24.001  1.00  0.00           H  )
  warnings.warn("WARNING: two consecutive residues with same number (%s, %s)" % (atom, self._current_residue.atoms[-1]))
/Users/hannahbaumann/.local/share/mamba/envs/openfe_env/lib/python3.12/site-packages/openmm/app/internal/pdbstructure.py:441: UserWarning: WARNING: two consecutive residues with same number (ATOM   3212  N   GLU A 211       9.982   5.977 -38.597  1.00  0.00           N  , HETATM 3211  H3  ACE A 211      11.950   4.338 -37.817  1.00  0.00           H  )
  warnings.warn("WARNING: two consecutive residues with same number (%s, %s)" % (atom, self._current_residue.atoms[-1]))
/Users/hannahbaumann/.local/share/mamba/envs/ope

Minimizing energy...
Potential energy after minimisation: -907061.7 kJ/mol
Running short NPT (500 steps)...


## 4. Set up and run the RBFE campaign

At this point the membrane-specific work is done: `complex_equil.pdb` is a 
fully built and solvated system with box vectors in its 
[`CRYST1`](https://www.wwpdb.org/documentation/file-format-content/format33/sect8.html#CRYST1) 
record (patched in section 2a). From here, planning, running, and analysing a 
membrane RBFE campaign follows the same steps as any other OpenFE RBFE campaign.

For the full campaign setup, please refer to the 
[membrane RBFE example notebook](https://github.com/OpenFreeEnergy/ExampleNotebooks/blob/main/membranes/rbfe_membrane_protein.ipynb), 
which covers ligand preparation, network planning, and analysis. The only 
membrane-specific difference is a single CLI flag when planning:

```bash
openfe plan-rbfe-network \
    --protein-membrane complex_equil.pdb \
    --molecules ligands_am1bcc.sdf \
    --output-dir rbfe_network/
```

<div class="alert alert-block alert-info">
    <b>Note:</b> Membrane support on the CLI requires 
    <code>openfe ≥ 1.11</code>, which introduced the 
    <code>--protein-membrane</code> flag.
</div>